# Diagnostics: why isn't the model improving?

Following `docs/diagnostic-prompt.md`. Diagnosis before fixes. Logic lives in `src/nfl_ats/diagnostics.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nfl_ats.data import HISTORY_SEASONS, PRIOR_SEASON, check_spread_sign, load_schedules, regular_season
from nfl_ats.diagnostics import (Z_80_POWER, accuracy_mde_grid, feature_health, leaky_walk_forward, lookup_games,
                                 margin_walk_forward, paired_mse_diff, paired_power, residual_correlations,
                                 row_counts, scaler_fit_check)
from nfl_ats.evaluate import BREAKEVEN, TEST_SEASONS, paired_diff, score
from nfl_ats.features import load_team_games
from nfl_ats.model import fold_data, walk_forward
from nfl_ats.features import FEATURES
from nfl_ats.target import add_home_cover

games = add_home_cover(regular_season(load_schedules()))
team_games = load_team_games([PRIOR_SEASON] + HISTORY_SEASONS)
FIG_DIR = "../reports/figures"

## Step 1. Rule out bugs

### 1a. Spread sign and target on games you can look up
`spread_line` > 0 means the home team is favored. `home_cover` = 1 if margin > spread_line, and NaN for a push.

In [ ]:
check_spread_sign(games)  # favorites win > 60%, spread rises with margin
lookup_games(games, ["2016_01_CAR_DEN", "2023_02_MIN_PHI", "2023_03_DAL_ARI", "2023_03_HOU_JAX"])

### 1b. What leakage looks like
The same pipeline, but season-to-date stats *include* the game being predicted (no `shift(1)`).

In [ ]:
honest = walk_forward(games, team_games, FEATURES, TEST_SEASONS)
leaky = leaky_walk_forward(games, team_games, TEST_SEASONS)
pd.concat({"honest v1": score(honest["home_cover"], honest["p"]),
           "LEAKY": score(leaky["home_cover"], leaky["p"])}, axis=1).round(4)

In [ ]:
# Where does the leak help most? Early weeks, when one game is a big share of the season-to-date average.
week = games.set_index("game_id")["week"]
def by_week(preds):
    d = preds.assign(wk=pd.cut(preds["game_id"].map(week), [0, 4, 9, 18], labels=["wk 1-4", "wk 5-9", "wk 10-18"]))
    return d.groupby("wk", observed=True).apply(lambda x: ((x["p"] > 0.5) == x["home_cover"]).mean(), include_groups=False)
pd.DataFrame({"honest": by_week(honest), "leaky": by_week(leaky),
              "games": honest["game_id"].map(week).pipe(pd.cut, [0, 4, 9, 18]).value_counts(sort=False).to_numpy()}).round(3)

### 1c. Row counts through the pipeline (fold: train 2016–2024, test 2025)

In [ ]:
row_counts(team_games, list(range(2016, 2025)), 2025)

### 1d. Feature health: NaN, constant, Week 1

In [ ]:
train, test = fold_data(games, team_games, list(range(2016, 2025)), 2025)
feature_health(pd.concat([train, test])).round(4)

### 1e. The scaler is fit on training rows only
A gap of exactly 0 against the training mean (and not against the test mean) proves the scaler only saw training rows.

In [ ]:
scaler_fit_check(games, team_games, TEST_SEASONS)

**Step 1 verdict: no bugs found.**
- **Target:** the hand-checkable games match the box scores.
- **Leakage demo:** a missing `shift(1)` turns 51.3% into **57.2% [54.9, 59.5]**, a fake "edge" whose CI clears 52.4%, with probability SD 0.114 vs. 0.027. This shows what a leak looks like. It is *not* the proof of no leakage, since leaks inflate scores and subtle ones can be quiet. That proof is the unit tests (`shift(1)`, fold isolation) plus the scaler check below.
- **Row counts:** they reconcile. 2,639 games − 65 pushes = 2,303 train + 271 test.
- **Features:** no NaNs, and Week 1 is filled by the prior. `rest_diff` is ~0 in Week 1 by design: everyone is rested.
- **Scaler:** it matches the training means exactly in every fold.

The model isn't failing because of a pipeline bug.

## Step 2. The noise floor: what improvement could we even detect?

**MDE** (minimum detectable effect) = the smallest *true* difference we'd catch 80% of the time with a 5% test = 2.8 × SE. In a paired comparison, only games where the two models pick differently move accuracy, so the accuracy SE ≈ √(disagreement rate / n).

In [ ]:
experiments = {
    "#4 +spread": walk_forward(games, team_games, FEATURES + ["spread_line"], TEST_SEASONS),
    "#5 garbage filter": pd.read_parquet("../data/processed/exp5_garbage_preds.parquet"),
    "#6 opponent adj": walk_forward(games, team_games, FEATURES, TEST_SEASONS, opponent_adjust=True),
}
y = honest["home_cover"]
rows = {}
for name, preds in experiments.items():
    pw = paired_power(y, honest["p"], preds["p"])
    rows[name] = {"disagreement": pw.loc["accuracy", "disagreement_rate"],
                  "acc_observed": pw.loc["accuracy", "observed_diff"], "acc_MDE": pw.loc["accuracy", "mde_80"],
                  "logloss_observed": pw.loc["log_loss", "observed_diff"], "logloss_MDE": pw.loc["log_loss", "mde_80"]}
pd.DataFrame(rows).T.round(4)

In [ ]:
# Accuracy MDE vs. how often a new model disagrees with v1 (n = 1,828 games)
accuracy_mde_grid(len(honest)).rename_axis("disagreement rate").round(4)

In [ ]:
# Is v1 even distinguishable from "always say 50%"? (new = v1, base = constant 0.5)
paired_diff(y, np.full(len(y), 0.5), honest["p"]).loc[["log_loss", "brier"]].round(4)

In [ ]:
# The bigger question: how good must a model TRULY be for 1,828 games to prove it beats 52.4%?
n = len(honest)
need_true = BREAKEVEN + Z_80_POWER * np.sqrt(BREAKEVEN * (1 - BREAKEVEN) / n)
games_needed = {f"true {t:.0%}": Z_80_POWER**2 * t * (1 - t) / (t - BREAKEVEN) ** 2 for t in [0.53, 0.54, 0.55, 0.56]}
print(f"With n = {n}, a model must truly hit {need_true:.1%} to be proven profitable 80% of the time.")
pd.Series(games_needed, name="games needed").to_frame().assign(seasons=lambda d: d["games needed"] / 272).round(0)

**Step 2 verdict:**
- **Our paired tests can see about 1–2.4 accuracy points.** That depends on how often the models disagree (2% to 13% here). The observed effects (0.0, −0.4, +0.2) are far below that, so accuracy could never have confirmed them.
- **Log loss is the more sensitive test.** Exp #6's log-loss gain (−0.0006) is about half its MDE (0.0011). That isn't detectable, but it's closer than accuracy ever gets. **Judge experiments on paired log loss, not accuracy.**
- **v1 vs. "always 50%":** +0.0015 [−0.0012, +0.0043]. We can't tell v1 apart from knowing nothing.
- **The hard ceiling:** to *prove* profit with 1,828 games, a model must truly hit about **55.7%**. A genuinely good 54% model would need about 27 seasons to prove. **Realistic edges are invisible in our sample by accuracy.** That's partly a measurement problem, not only a model problem.

## Step 3. Where is the signal?

The key question: do our features know anything the closing spread doesn't? We test it on **margin**, a richer target than cover/no-cover, so it's more sensitive.

### 3a. Predict the final margin, walk-forward, out of sample
Pushes are kept, since margin doesn't need them excluded. `spread_as_is` sets margin = spread, with nothing fit.

In [ ]:
margins = margin_walk_forward(games, team_games, TEST_SEASONS)
pd.DataFrame({
    "fit spread-only vs. spread as-is": paired_mse_diff(margins["margin"], margins["spread_as_is"], margins["spread_only"]),
    "spread + features vs. spread-only": paired_mse_diff(margins["margin"], margins["spread_only"], margins["spread_plus_features"]),
}).T.round(4)

In [ ]:
# Same comparison per season: a real signal should help in most seasons, not flip sign.
pd.DataFrame({s: paired_mse_diff(g["margin"], g["spread_only"], g["spread_plus_features"])
              for s, g in margins.groupby("season")}).T[["pct", "ci_low", "ci_high"]].round(4)

### 3b. Does any feature correlate with what the spread got wrong?
Residual = margin − spread_line. A feature with information the line misses correlates with the residual consistently across seasons.

In [ ]:
rc = residual_correlations(margins)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharey=True)
for ax, feature in zip(axes, FEATURES):
    d = rc[(rc["feature"] == feature) & (rc["season"] != "all")]
    ax.errorbar(d["season"].astype(int), d["r"], yerr=[d["r"] - d["ci_low"], d["ci_high"] - d["r"]],
                fmt="o", ms=5, capsize=3, lw=1.5, color="steelblue")
    pooled = rc[(rc["feature"] == feature) & (rc["season"] == "all")].iloc[0]
    ax.axhspan(pooled["ci_low"], pooled["ci_high"], color="steelblue", alpha=0.12)
    ax.axhline(0, color="gray", lw=1)
    ax.set(title=f"{feature}\npooled r = {pooled['r']:+.3f}", xlabel="Season")
    ax.tick_params(axis="x", labelrotation=45)
axes[0].set_ylabel("Correlation with margin − spread")
fig.suptitle("No feature reliably predicts what the spread misses: signs flip season to season", y=1.02)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/residual_correlations.png", dpi=150, bbox_inches="tight")

In [ ]:
# The Step 1 lead (better early-season accuracy): do features predict the residual in weeks 1-4?
early = margins.assign(period=margins["week"].le(4).map({True: "wk 1-4", False: "wk 5+"}))
residual_correlations(early, by="period").query("period != 'all'").pivot(index="feature", columns="period", values=["r", "ci_low", "ci_high"]).round(3)

**Step 3 verdict (Josh predicted a 1% error reduction), corrected after the skeptic review:**
- **Adding our features makes out-of-sample margin predictions slightly *worse*:** +0.35% MSE [−0.18%, +0.91%]. Even fitting a spread-only model is slightly worse than the spread as-is (+0.16%). With no signal, estimating coefficients adds noise (the expected null value is about +0.29%).
- **Per season the effect flips sign** (−0.7% to +1.2%).
- **Residual correlations, controlling for spread (partial r):** off EPA +0.016 [−0.030, +0.061], def EPA +0.014, pace −0.029, rest −0.007. Every CI includes 0. Signs flip season to season.
- **Off EPA in 2024 (r = +0.185, p = 0.002) is the one exception.** That's rarer than routine chance (about a 6% event across 28 feature-seasons), but it fails the stability test: the other six seasons range from −0.03 to +0.05.
- **Early-season lead: not supported.** No feature's CI excludes 0 in weeks 1–4, but with n = 446 effects up to about r = 0.15 can't be ruled out either.

**What we can honestly say:** no *detectable* incremental signal. As a rough guide, a correlation r with the residual maps to about ½ + arcsin(r)/π ATS. The upper CI bounds (about 0.06–0.07) correspond to about **52.4% at most**, so any linear signal in these four features is at most breakeven-sized: too small to profit, and too small for this sample to see. The margin-MSE test itself has little power (it detects a breakeven-sized effect only about 11% of the time in the skeptic's simulation), so the CIs above carry the evidence, not the MSE comparison.

**What we did *not* test:** nonlinear effects, interactions (e.g. model-vs-line disagreement, spread size), key-number structure (3 and 7), or subsets. So the claim is limited to *these four season-to-date features, entered linearly*. Finding signal needs **different information** (Step 6), or a different way of using it.